# CS224N · Lecture 5 — Language Models and Recurrent Neural Networks

**Slides:** [cs224n-spr2024-lecture05-rnnlm.pdf](https://web.stanford.edu/class/cs224n/slides/cs224n-spr2024-lecture05-rnnlm.pdf)  
**Prerequisites:** L3 (backprop), L2 (cross-entropy). The L4 notebook's habits (gradient checks, error analysis) are used throughout.

### Why this lecture matters
The slides call **language modeling** *"the most important concept in the class! It leads to BERT, GPT-3 and ChatGPT!"* Every modern LLM is a language model, trained with exactly the objective in this notebook: predict the next token, minimize cross-entropy. Only the architecture changes (RNN here → Transformer in L8).

### What you will build
| § | Topic | You will implement |
|---|---|---|
| 1 | A bit more about neural nets: regularization, dropout, vectorization, initialization, optimizers | Dropout layer; the symmetry problem; optimizer zoo |
| 2 | Language modeling; n-gram LMs; sparsity, smoothing, backoff | n-gram LMs from counts; text generation |
| 3 | Fixed-window neural LM (Bengio et al. 2003) | Window LM in NumPy |
| 4 | **Recurrent neural networks**; training; **backprop through time** | RNN LM with hand-written BPTT + gradient check |
| 5 | Generation and evaluation (perplexity) | Sampling with temperature; perplexity comparisons |
| 6 | **Vanishing and exploding gradients**; gradient clipping | Spectral-radius experiments; the "cliff" |
| 7 | Other uses of RNNs; RNN ≠ LM | A diagnostic probe of the hidden state |
| 8 | PyTorch versions (+ optional char-RNN on Shakespeare) | `nn.RNN` LM |

### The running experiment: long-distance agreement
To *measure* what different language models can and can't do, we use a small artificial language with **subject–verb agreement across long distances**:

> *the **dogs** that the cat saw near the old farmer **sleep** .*  
> *the **dog** behind the keys that the authors liked **sleeps** .*

To predict the right verb form, a model must remember the number (singular/plural) of the subject across a relative clause or prepositional phrases, which often contain **attractor** nouns of the opposite number. This is a classic test of LMs' syntactic abilities (Linzen, Dupoux & Goldberg 2016; Gulordava et al. 2018).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import random, math, time, collections

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(0)
random.seed(0)

def softmax(x, axis=-1):
    z = x - x.max(axis=axis, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=axis, keepdims=True)

---
## 1. A bit more about neural networks

Modern neural networks (especially language models) are **enormous**, with billions of parameters. But building large neural networks wasn't easy or obvious: it took a long time and much work to make deep networks practical (e.g., Bengio et al. 2006 on greedy layer-wise pretraining). This section collects the practical tools that made it work.

### 1.1 Regularization
We have models with many parameters, so we need **regularization**. A full loss function includes regularization over all parameters $\theta$, e.g., **L2 regularization**:
$$J(\theta) = \frac{1}{N}\sum_{i=1}^{N} -\log\left(\frac{e^{f_{y_i}}}{\sum_{c=1}^{C}e^{f_c}}\right) + \lambda\sum_k\theta_k^2$$
* **Classic view:** regularization prevents **overfitting** when we have a lot of features (or a very powerful/deep model).
* **Now:** regularization produces models that **generalize well when we have a "big" model**. We do not care that our models overfit on the training data, even though they are hugely overfit. (🔎 This connects to "double descent": very overparameterized models can generalize *better*; see Belkin et al. 2019; Nakkiran et al. 2019.)

### 1.2 Dropout (Srivastava, Hinton, Krizhevsky, Sutskever & Salakhutdinov 2012/2014)
* **During training:** for each data point, each time, randomly set each input to 0 with probability $p$ (the "dropout ratio", often $p=0.5$, except $p\approx0.15$ for the input layer), via a random dropout mask.
* **During testing:** multiply all weights by $1-p$; no other dropout. (This keeps the expected input to each unit the same as in training.)

**Why does it work?** It prevents **feature co-adaptation**: a feature can't be useful only in the presence of particular other features.
* In a single layer it's a middle ground between Naïve Bayes (all feature weights set independently) and logistic regression (weights set in the context of all others).
* It can be thought of as a form of **model bagging** (an ensemble of the $2^n$ thinned networks that share weights).
* Nowadays it's usually thought of as a strong, **feature-dependent regularizer** (Wager, Wang & Liang 2013).

🔎 **Inverted dropout.** Real frameworks (including PyTorch's `nn.Dropout`) do the equivalent thing the other way around: at *training* time they scale the kept activations by $\frac{1}{1-p}$, so that *test time needs no change at all*. Same expectations, simpler deployment.

In [ ]:
class Dropout:
    # Inverted dropout: scale at train time so test time is the identity.
    def __init__(self, p):
        self.p = p
    def forward(self, x, train=True, rng=rng):
        if not train or self.p == 0:
            return x
        self.mask = (rng.random(x.shape) >= self.p) / (1 - self.p)
        return x * self.mask
    def backward(self, dout):
        return dout * self.mask          # dropped units get no gradient

x = np.ones(100_000)
drop = Dropout(0.5)
y = drop.forward(x)
print("fraction zeroed:", (y == 0).mean().round(3), " mean of output:", y.mean().round(3), "(expectation preserved = 1)")

# Equivalence with the slides' formulation: no scaling in training, multiply by (1-p) at test time.
w = rng.normal(size=4); xin = rng.normal(size=4); p = 0.5
masks = rng.random((200_000, 4)) >= p
train_avg_slides = np.mean((masks * xin) @ w)        # average output of the thinned nets (no scaling)
print(f"slides' test-time rule (1-p) * w.x = {(1 - p) * w @ xin:.4f}   vs  average over random masks = {train_avg_slides:.4f}")

### 1.3 "Vectorization"
E.g., looping over word vectors vs. concatenating them all into one large matrix and multiplying the softmax weights with that matrix. The slide reports **639 µs per loop vs. 53.8 µs** with a single $C\times N$ matrix. **Matrices are awesome!!! Always try to use vectors and matrices rather than for loops!** The speed gain goes from 1 to 2 orders of magnitude with GPUs. Let's reproduce it:

In [ ]:
N, d, C = 500, 300, 5
W = rng.normal(size=(C, d))
wordvectors = [rng.normal(size=d) for _ in range(N)]
wordvectors_matrix = np.stack(wordvectors, axis=1)        # d x N

def loop_version():
    return [W @ wordvectors[i] for i in range(N)]
def matrix_version():
    return W @ wordvectors_matrix                         # one C x N result

for f in (loop_version, matrix_version):
    t = time.perf_counter()
    for _ in range(200):
        f()
    print(f"{f.__name__:15s} {(time.perf_counter() - t) / 200 * 1e6:8.1f} µs per call")
print("same result:", np.allclose(np.stack(loop_version(), axis=1), matrix_version()))

### 1.4 Parameter initialization
* You normally must initialize weights to **small random values** (not zero matrices!) to **avoid symmetries** that prevent learning/specialization. (Demo below.)
* Initialize hidden-layer biases to 0, and output (or reconstruction) biases to the optimal value if the weights were 0 (e.g., the mean target, or the inverse sigmoid of the mean target).
* Initialize all other weights $\sim\text{Uniform}(-r, r)$, with $r$ chosen so numbers get neither too big nor too small. (Later, the need for careful init is reduced by **layer normalization**, L8.)
* **Xavier initialization** has variance inversely proportional to fan-in $n_{\text{in}}$ and fan-out $n_{\text{out}}$:
$$\text{Var}(W_{ij}) = \frac{2}{n_{\text{in}} + n_{\text{out}}} \qquad\Big(\text{uniform version: } r = \sqrt{6/(n_{\text{in}}+n_{\text{out}})}\Big)$$

In [ ]:
# The symmetry problem: with identical initial weights, all hidden units get identical gradients forever.
def train_two_layer(W1, steps=200, lr=0.5):
    X = rng.normal(size=(64, 3)); y = (X[:, 0] * X[:, 1] > 0).astype(float)
    W1 = W1.copy(); w2 = np.full(W1.shape[0], 0.1)
    for _ in range(steps):
        H = np.tanh(X @ W1.T)
        p = 1 / (1 + np.exp(-(H @ w2)))
        g = (p - y) / len(y)
        dW1 = ((g[:, None] * w2[None]) * (1 - H ** 2)).T @ X
        w2 -= lr * (H.T @ g); W1 -= lr * dW1
    return W1

W_sym = train_two_layer(np.full((4, 3), 0.3))
W_rand = train_two_layer(rng.normal(scale=0.5, size=(4, 3)))
print("constant init -> hidden units after training (rows):\n", W_sym.round(3), "\n  all identical: the 4 units are really 1 unit")
print("random init   -> hidden units after training (rows):\n", W_rand.round(3))

### 1.5 Optimizers
* Usually, **plain SGD will work just fine!** But getting good results often requires **hand-tuning the learning rate**, e.g., start it higher and halve it every $k$ epochs.
* For more complex nets, or to avoid worry, try more sophisticated **adaptive** optimizers that scale the adjustment to individual parameters by an accumulated gradient. These give **per-parameter learning rates**:

| Optimizer | Update (with $g$ = gradient) | Note from the slide |
|---|---|---|
| **Adagrad** | $G \mathrel{+}= g^2$; $\ \theta \mathrel{-}= \alpha\, g/\sqrt{G+\epsilon}$ | simplest member of the family, but tends to **"stall early"** ($G$ only grows) |
| **RMSprop** | $G \leftarrow \beta G + (1-\beta) g^2$; $\ \theta \mathrel{-}= \alpha\, g/\sqrt{G+\epsilon}$ | fixes stalling with a moving average |
| **Adam** | momentum $m$ + RMSprop $v$ + bias correction (see L2 notebook) | **a fairly good, safe place to begin** |
| **AdamW** | Adam + *decoupled* weight decay $\theta \mathrel{-}= \alpha\lambda\theta$ | default for Transformers |
| **NAdamW** | AdamW + Nesterov momentum | can be better with word vectors and for speed |

Start them with an initial learning rate around **0.001**. Many have other hyperparameters.

In [ ]:
def rosenbrock_grad(th):
    x, y = th
    return np.array([-2 * (1 - x) - 400 * x * (y - x ** 2), 200 * (y - x ** 2)])

def optimize(name, lr, steps=3000):
    th = np.array([-1.5, 2.0]); s = {"G": np.zeros(2), "m": np.zeros(2), "v": np.zeros(2)}
    for t in range(1, steps + 1):
        g = rosenbrock_grad(th)
        if name == "SGD":
            th = th - lr * g
        elif name == "Adagrad":
            s["G"] += g ** 2; th = th - lr * g / np.sqrt(s["G"] + 1e-8)
        elif name == "RMSprop":
            s["G"] = 0.9 * s["G"] + 0.1 * g ** 2; th = th - lr * g / np.sqrt(s["G"] + 1e-8)
        elif name == "Adam":
            s["m"] = 0.9 * s["m"] + 0.1 * g; s["v"] = 0.999 * s["v"] + 0.001 * g ** 2
            th = th - lr * (s["m"] / (1 - 0.9 ** t)) / (np.sqrt(s["v"] / (1 - 0.999 ** t)) + 1e-8)
    return th

for name, lr in [("SGD", 1e-3), ("Adagrad", 0.1), ("RMSprop", 1e-2), ("Adam", 1e-2)]:
    th = optimize(name, lr)
    print(f"{name:8s} lr={lr:<6} -> theta = {th.round(4)}   distance to optimum (1,1) = {np.linalg.norm(th - 1):.4f}")

---
## 2. A new NLP task: language modeling

**Language modeling** is the task of predicting what word comes next: *the students opened their \_\_\_\_\_\_* (books? laptops? exams? minds?).

More formally: given a sequence of words $x^{(1)}, x^{(2)}, \dots, x^{(t)}$, compute the probability distribution of the next word $x^{(t+1)}$:
$$P\big(x^{(t+1)} \mid x^{(t)}, \dots, x^{(1)}\big)$$
where $x^{(t+1)}$ can be any word in the vocabulary $V = \{w_1, \dots, w_{|V|}\}$. A system that does this is called a **language model**.

You can also think of a language model as a system that **assigns a probability to a piece of text**. By the chain rule of probability:
$$P\big(x^{(1)}, \dots, x^{(T)}\big) = P(x^{(1)})\times P(x^{(2)}\mid x^{(1)})\times\cdots = \prod_{t=1}^{T}\underbrace{P\big(x^{(t)}\mid x^{(t-1)},\dots,x^{(1)}\big)}_{\text{this is what our LM provides}}$$

You use language models every day: phone keyboard prediction, search autocomplete, and of course chatbots.

### 2.1 n-gram language models
**Question:** How to learn a language model? **Answer (pre-deep learning):** learn an **n-gram** LM.

**Definition:** an **n-gram** is a chunk of $n$ consecutive words.
* unigrams: "the", "students", "opened", "their"
* bigrams: "the students", "students opened", "opened their"
* trigrams: "the students opened", "students opened their"
* four-grams: "the students opened their"

**Idea:** collect statistics about how frequent different n-grams are, and use these to predict the next word.

First we make a **Markov assumption**: $x^{(t+1)}$ depends only on the preceding $n-1$ words:
$$P\big(x^{(t+1)}\mid x^{(t)},\dots,x^{(1)}\big) \approx P\big(x^{(t+1)}\mid x^{(t)},\dots,x^{(t-n+2)}\big) \quad\text{(assumption)}$$
$$= \frac{P\big(x^{(t+1)}, x^{(t)},\dots,x^{(t-n+2)}\big)}{P\big(x^{(t)},\dots,x^{(t-n+2)}\big)} = \frac{\text{prob of an } n\text{-gram}}{\text{prob of an } (n-1)\text{-gram}}\quad\text{(definition of conditional prob)}$$

**How do we get these probabilities? By counting** in a large corpus:
$$\approx \frac{\text{count}\big(x^{(t+1)}, x^{(t)},\dots,x^{(t-n+2)}\big)}{\text{count}\big(x^{(t)},\dots,x^{(t-n+2)}\big)}\quad\text{(statistical approximation)}$$

**Example (4-gram LM).** *as the proctor started the clock, the students opened their \_\_\_\_*. We discard everything but "students opened their". Suppose in the corpus:

In [ ]:
counts = {("students", "opened", "their"): 1000,
          ("students", "opened", "their", "books"): 400,
          ("students", "opened", "their", "exams"): 100}
ctx = ("students", "opened", "their")
for w in ["books", "exams"]:
    print(f"P({w} | students opened their) = {counts[ctx + (w,)]} / {counts[ctx]} = {counts[ctx + (w,)] / counts[ctx]}")
print("\nShould we have discarded the 'proctor' context? (It makes 'exams' much more likely!)")

### 2.2 Problems with n-gram LMs
**Sparsity problem 1:** What if "students opened their $w$" never occurred in the data? Then $w$ has probability 0!
* *(Partial) solution:* add a small $\delta$ to the count for every $w\in V$. This is called **smoothing**:
$P_{\text{add-}\delta}(w\mid\text{ctx}) = \frac{\text{count}(\text{ctx}, w) + \delta}{\text{count}(\text{ctx}) + \delta|V|}$

**Sparsity problem 2:** What if "students opened their" never occurred? Then we can't calculate a probability for *any* $w$!
* *(Partial) solution:* just condition on "opened their" instead. This is called **backoff**.

**Increasing $n$ makes sparsity problems worse.** Typically, we can't have $n$ bigger than 5.

**Storage problem:** we need to store counts for all n-grams we saw in the corpus. Increasing $n$ or the corpus increases model size.

🔎 **What practitioners actually used:** **interpolation** (mix estimates of different orders: $\lambda_3 P_3 + \lambda_2 P_2 + \lambda_1 P_1$), and especially **Kneser–Ney smoothing**, which bases the lower-order distribution on how many *different contexts* a word appears in ("Francisco" is frequent but only after "San"). KenLM-style 5-gram Kneser–Ney models were the backbone of speech recognition and SMT until ~2015.

### 2.3 Our toy language
Now let's generate the agreement corpus and train n-gram LMs on it.

In [ ]:
NOUNS = [("dog", "dogs"), ("cat", "cats"), ("pilot", "pilots"), ("farmer", "farmers"), ("author", "authors"), ("key", "keys")]
VERBS = [("sleeps", "sleep"), ("laughs", "laugh"), ("waits", "wait"), ("smiles", "smile")]
TRANS = ["saw", "liked", "chased", "met", "helped"]
ADJ = ["old", "young", "tall", "small", "happy"]
PREP = ["near", "behind", "with"]

def gen_sentence(max_mods=3, rng=random):
    # Returns (words, subject_position, verb_position, subject_is_plural, verb_pair).
    plural = rng.random() < 0.5
    w = ["the"]
    if rng.random() < 0.3:
        w.append(rng.choice(ADJ))
    w.append(rng.choice(NOUNS)[plural]); subj = len(w) - 1
    for _ in range(rng.choice(range(max_mods + 1))):
        attractor_plural = rng.random() < 0.5          # nouns inside modifiers: random number
        if rng.random() < 0.5:                         # relative clause: "that the cat saw"
            w += ["that", "the"] + ([rng.choice(ADJ)] if rng.random() < 0.3 else [])
            w += [rng.choice(NOUNS)[attractor_plural], rng.choice(TRANS)]
        else:                                          # prepositional phrase: "near the old keys"
            w += [rng.choice(PREP), "the"] + ([rng.choice(ADJ)] if rng.random() < 0.3 else [])
            w.append(rng.choice(NOUNS)[attractor_plural])
    vpair = rng.choice(VERBS)
    w.append(vpair[plural]); verb = len(w) - 1
    if rng.random() < 0.3:
        w.append("quietly")
    w.append(".")
    return w, subj, verb, plural, vpair

random.seed(0)
train_data = [gen_sentence() for _ in range(6000)]
test_data = [gen_sentence() for _ in range(1500)]
vocab = ["<s>", "</s>"] + sorted({w for s, *_ in train_data + test_data for w in s})
w2i = {w: i for i, w in enumerate(vocab)}
V = len(vocab)
for s, sp, vp, pl, _ in train_data[:5]:
    print(" ".join(s).ljust(70), f"subject='{s[sp]}' verb='{s[vp]}' distance={vp - sp}")
dist = collections.Counter(min(vp - sp, 12) for _, sp, vp, _, _ in test_data)
print(f"\n|V| = {V}; test distances (12 = 12+): {dict(sorted(dist.items()))}")

In [ ]:
class NGramLM:
    def __init__(self, sentences, n, delta=0.01):
        self.n, self.delta = n, delta
        self.counts = collections.defaultdict(collections.Counter)
        for s, *_ in sentences:
            toks = ["<s>"] * (n - 1) + s + ["</s>"]
            for t in range(n - 1, len(toks)):
                self.counts[tuple(toks[t - n + 1:t])][toks[t]] += 1

    def prob(self, ctx, w):
        # Add-delta smoothed P(w | last n-1 words of ctx).
        c = self.counts.get(tuple(ctx[len(ctx) - self.n + 1:]) if self.n > 1 else (), {})
        total = sum(c.values()) if c else 0
        return (c.get(w, 0) + self.delta) / (total + self.delta * V)

    def sentence_logprob(self, s):
        toks = ["<s>"] * (self.n - 1) + s + ["</s>"]
        return sum(math.log(self.prob(toks[:t], toks[t])) for t in range(self.n - 1, len(toks))), len(s) + 1

def perplexity_ngram(lm, sentences):
    lp, n = zip(*(lm.sentence_logprob(s) for s, *_ in sentences))
    return math.exp(-sum(lp) / sum(n))

def bucket(d):
    return "1" if d == 1 else ("2-6" if d <= 6 else ("7-10" if d <= 10 else "11+"))

def agreement_ngram(lm, sentences):
    acc = collections.defaultdict(list)
    for s, sp, vp, pl, vpair in sentences:
        ctx = ["<s>"] * (lm.n - 1) + s[:vp]
        acc[bucket(vp - sp)].append(lm.prob(ctx, vpair[pl]) > lm.prob(ctx, vpair[1 - pl]))
    return {k: round(float(np.mean(v)), 3) for k, v in sorted(acc.items())}

ngrams = {n: NGramLM(train_data, n) for n in [2, 3, 4]}
for n, lm in ngrams.items():
    print(f"{n}-gram: test perplexity {perplexity_ngram(lm, test_data):.3f}   agreement accuracy by distance: {agreement_ngram(lm, test_data)}")

**Read this carefully.** When the verb is right after the subject (distance 1), the n-gram models are nearly perfect: the context *contains* the subject. Beyond that, they're at chance (~50%) or even **below chance**. Below chance happens because the words just before the verb are often an **attractor noun** of the opposite number (*"…near the old **keys**"*), and the n-gram model copies its number. Humans make the same "agreement attraction" error (*"the key to the cabinets **are**…"*).

### 2.4 Generating text with an n-gram LM
You can also use an LM to **generate text**: condition on the context, get a probability distribution, **sample**, append, repeat. The slide's trigram model trained on Reuters produces text that is *"surprisingly grammatical! …but incoherent. We need to consider more than three words at a time if we want to model language well."* In our toy language, incoherence shows up as **agreement violations**:

In [ ]:
def generate_ngram(lm, max_len=30, rng=random):
    toks = ["<s>"] * (lm.n - 1)
    while len(toks) < max_len + lm.n:
        ctx = tuple(toks[len(toks) - lm.n + 1:])
        words, weights = zip(*lm.counts[ctx].items())          # sample from the unsmoothed counts
        w = rng.choices(words, weights=weights)[0]
        if w == "</s>":
            break
        toks.append(w)
    return toks[lm.n - 1:]

def agreement_violation(sent):
    # Check the main verb agrees with the first noun (our grammar's subject). None if unparseable.
    sg_n, pl_n = {n[0] for n in NOUNS}, {n[1] for n in NOUNS}
    sg_v, pl_v = {v[0] for v in VERBS}, {v[1] for v in VERBS}
    subj = next((w for w in sent if w in sg_n | pl_n), None)
    verb = next((w for w in sent if w in sg_v | pl_v), None)
    if subj is None or verb is None:
        return None
    return (subj in pl_n) != (verb in pl_v)

random.seed(1)
samples = [generate_ngram(ngrams[3]) for _ in range(500)]
for s in samples[:5]:
    print("  ", " ".join(s))
viol = [v for v in map(agreement_violation, samples) if v is not None]
print(f"\ntrigram samples with an agreement violation: {np.mean(viol):.1%}")

---
## 3. How to build a neural language model? A fixed-window neural LM

Recall the LM task: input a sequence of words, output a probability distribution of the next word. How about a **window-based neural model**, like the NER classifier from Lecture 2? (Approximately Bengio et al. 2000/2003, *A Neural Probabilistic Language Model*.)

*as the proctor started the clock* ~~discard~~ | **the students opened their** (fixed window) → ?

* words / one-hot vectors: $x^{(1)}, x^{(2)}, x^{(3)}, x^{(4)}$
* concatenated word embeddings: $e = [e^{(1)}; e^{(2)}; e^{(3)}; e^{(4)}]$
* hidden layer: $h = f(We + b_1)$
* output distribution: $\hat y = \text{softmax}(Uh + b_2)\in\mathbb{R}^{|V|}$

**Improvements over n-gram LM:**
* No sparsity problem (similar words share statistical strength through their embeddings)
* Don't need to store all observed n-grams

**Remaining problems:**
* The fixed window is **too small**, and enlarging the window enlarges $W$. The window can never be large enough!
* $x^{(1)}$ and $x^{(2)}$ are multiplied by completely different weights in $W$. **No symmetry** in how the inputs are processed.

**We need a neural architecture that can process any length input.**

In [ ]:
def windows_lm(sentences, k=3):
    X, Y = [], []
    for s, *_ in sentences:
        toks = ["<s>"] * k + s + ["</s>"]
        for t in range(k, len(toks)):
            X.append([w2i[w] for w in toks[t - k:t]]); Y.append(w2i[toks[t]])
    return np.array(X), np.array(Y)

class Adam:
    def __init__(self, params, lr=2e-3, b1=0.9, b2=0.999, eps=1e-8):
        self.lr, self.b1, self.b2, self.eps, self.t = lr, b1, b2, eps, 0
        self.m = {k: np.zeros_like(v) for k, v in params.items()}
        self.v = {k: np.zeros_like(v) for k, v in params.items()}
    def step(self, params, grads):
        self.t += 1
        for k in params:
            self.m[k] = self.b1 * self.m[k] + (1 - self.b1) * grads[k]
            self.v[k] = self.b2 * self.v[k] + (1 - self.b2) * grads[k] ** 2
            params[k] -= self.lr * (self.m[k] / (1 - self.b1 ** self.t)) / (np.sqrt(self.v[k] / (1 - self.b2 ** self.t)) + self.eps)

def window_lm_fb(p, X, Y, need_grad=True):
    B, k = X.shape; d = p["E"].shape[1]
    e = p["E"][X].reshape(B, -1)
    z = e @ p["W"].T + p["b1"]; h = np.tanh(z)
    P = softmax(h @ p["U"].T + p["b2"], axis=1)
    loss = -np.log(P[np.arange(B), Y] + 1e-12).mean()
    if not need_grad:
        return loss, P
    G = P.copy(); G[np.arange(B), Y] -= 1; G /= B                 # softmax + CE: y_hat - y
    g = {"U": G.T @ h, "b2": G.sum(0)}
    dz = (G @ p["U"]) * (1 - h ** 2)
    g["W"], g["b1"] = dz.T @ e, dz.sum(0)
    g["E"] = np.zeros_like(p["E"]); np.add.at(g["E"], X.reshape(-1), (dz @ p["W"]).reshape(-1, d))
    return loss, g

K_WIN = 3
Xw_tr, Yw_tr = windows_lm(train_data, K_WIN)
Xw_te, Yw_te = windows_lm(test_data, K_WIN)
r = np.random.default_rng(0); d_e, H_w = 32, 64
p_win = {"E": r.normal(0, 0.1, (V, d_e)), "W": r.normal(0, 1 / np.sqrt(K_WIN * d_e), (H_w, K_WIN * d_e)), "b1": np.zeros(H_w),
         "U": r.normal(0, 1 / np.sqrt(H_w), (V, H_w)), "b2": np.zeros(V)}
opt = Adam(p_win)
for epoch in range(4):
    perm = r.permutation(len(Xw_tr))
    for kk in range(0, len(perm), 128):
        idx = perm[kk:kk + 128]
        _, g = window_lm_fb(p_win, Xw_tr[idx], Yw_tr[idx]); opt.step(p_win, g)
    print(f"epoch {epoch + 1}: test perplexity {math.exp(window_lm_fb(p_win, Xw_te, Yw_te, False)[0]):.3f}")

def agreement_window(p, sentences):
    acc = collections.defaultdict(list)
    for s, sp, vp, pl, vpair in sentences:
        ctx = (["<s>"] * K_WIN + s[:vp])[-K_WIN:]
        _, P = window_lm_fb(p, np.array([[w2i[w] for w in ctx]]), np.array([0]), False)
        acc[bucket(vp - sp)].append(P[0, w2i[vpair[pl]]] > P[0, w2i[vpair[1 - pl]]])
    return {k: round(float(np.mean(v)), 3) for k, v in sorted(acc.items())}
print("fixed-window (k=3) agreement accuracy by distance:", agreement_window(p_win, test_data))

As expected: a smoother model than the n-gram, but **still blind beyond its window**. To agree with a subject 8 words back, it would need $k\geq 9$, and a bigger $W$, and it would still fail at distance 15.

---
## 4. Recurrent neural networks (RNNs)

**Core idea: apply the same weights $W$ repeatedly.** A family of neural architectures with hidden states $h^{(0)}, h^{(1)}, \dots$, an input sequence of any length, and (optionally) an output at each step.

### 4.1 A simple RNN language model
$$\text{words / one-hot vectors: } x^{(t)}\in\mathbb{R}^{|V|} \qquad \text{word embeddings: } e^{(t)} = E\,x^{(t)}$$
$$\text{hidden states: } h^{(t)} = \sigma\!\left(W_h h^{(t-1)} + W_e e^{(t)} + b_1\right),\quad h^{(0)} \text{ is the initial hidden state}$$
$$\text{output distribution: } \hat y^{(t)} = \text{softmax}\!\left(U h^{(t)} + b_2\right)\in\mathbb{R}^{|V|}$$

**RNN advantages:**
* Can process **any length input**
* Computation for step $t$ can (in theory) use information from **many steps back**
* **Model size doesn't increase** for longer input context
* The **same weights** are applied on every timestep, so there is **symmetry** in how inputs are processed

**RNN disadvantages:**
* Recurrent computation is **slow** (step $t$ needs step $t-1$ first, so it can't be parallelized over time)
* In practice, it's **difficult to access information from many steps back** (§6)

### 4.2 Training an RNN language model
* Get a **big corpus** of text, a sequence of words $x^{(1)},\dots,x^{(T)}$.
* Feed it into the RNN-LM; compute the output distribution $\hat y^{(t)}$ **for every step $t$**, i.e., predict the probability distribution of *every word, given the words so far*.
* The **loss on step $t$** is the cross-entropy between the predicted distribution $\hat y^{(t)}$ and the true next word $y^{(t)}$ (one-hot for $x^{(t+1)}$):
$$J^{(t)}(\theta) = CE\big(y^{(t)}, \hat y^{(t)}\big) = -\sum_{w\in V} y_w^{(t)}\log\hat y_w^{(t)} = -\log\hat y^{(t)}_{x_{t+1}}$$
* Average this to get the **overall loss** for the entire training set:
$$J(\theta) = \frac{1}{T}\sum_{t=1}^{T}J^{(t)}(\theta)$$

At every step, the input is the **true** previous word from the corpus, not the model's own prediction. This is called **teacher forcing**.

However, computing the loss and gradients across the *entire corpus* at once is too expensive (memory-wise)! In practice, consider $x^{(1)},\dots,x^{(T)}$ to be a **sentence** (or a document). Recall SGD: compute the loss and gradients for a small chunk of data (actually, a **batch of sentences**), update, repeat.

### 4.3 Backpropagation for RNNs: backpropagation through time (BPTT)
**Question:** what's the derivative of $J^{(t)}(\theta)$ with respect to the **repeated** weight matrix $W_h$?

**Answer:**
$$\frac{\partial J^{(t)}}{\partial W_h} = \sum_{i=1}^{t}\frac{\partial J^{(t)}}{\partial W_h}\bigg|_{(i)}$$
**"The gradient w.r.t. a repeated weight is the sum of the gradient w.r.t. each time it appears."**

**Why?** The **multivariable chain rule**: for $f(x(t), y(t))$, $\frac{d}{dt}f = \frac{\partial f}{\partial x}\frac{dx}{dt} + \frac{\partial f}{\partial y}\frac{dy}{dt}$. Think of each use of $W_h$ as a separate variable $W_h|_{(i)}$ that happens to equal $W_h$, so $\frac{\partial W_h|_{(i)}}{\partial W_h} = 1$ and the chain rule sums the contributions.

**How do we compute it?** Backpropagate over timesteps $i = t, \dots, 0$, **summing gradients as you go**. This algorithm is called **backpropagation through time** (Werbos 1988, and others). In practice it's often **truncated** after ~20 timesteps for training-efficiency reasons.

Concretely, with $z^{(t)} = W_h h^{(t-1)} + W_e e^{(t)} + b_1$, $h^{(t)} = \tanh(z^{(t)})$, and $\delta_h^{(t)} = \partial J/\partial h^{(t)}$ (which collects the gradient from the output at step $t$ **and** from step $t+1$):
$$\delta_h^{(t)} = U^\top(\hat y^{(t)} - y^{(t)}) + W_h^\top\delta_z^{(t+1)}, \qquad \delta_z^{(t)} = \delta_h^{(t)}\odot(1 - h^{(t)2})$$
$$\frac{\partial J}{\partial W_h} = \sum_t \delta_z^{(t)}h^{(t-1)\top},\quad \frac{\partial J}{\partial W_e} = \sum_t \delta_z^{(t)}e^{(t)\top},\quad \frac{\partial J}{\partial U} = \sum_t(\hat y^{(t)} - y^{(t)})h^{(t)\top}$$

Let's implement this for mini-batches of padded sentences (a mask zeroes out the loss at padding positions):

In [ ]:
def batchify(sents):
    # Inputs <s> w1 .. wn ; targets w1 .. wn </s> ; mask marks real (non-padding) positions.
    L = max(len(s[0]) for s in sents) + 1
    X = np.zeros((len(sents), L), int); Y = np.zeros((len(sents), L), int); M = np.zeros((len(sents), L))
    for b, (s, *_) in enumerate(sents):
        ids = [w2i["<s>"]] + [w2i[w] for w in s] + [w2i["</s>"]]
        X[b, :len(ids) - 1] = ids[:-1]; Y[b, :len(ids) - 1] = ids[1:]; M[b, :len(ids) - 1] = 1
    return X, Y, M

def init_rnn(V, d=32, H=64, seed=0):
    r = np.random.default_rng(seed)
    return {"E": r.normal(0, 0.1, (V, d)), "We": r.normal(0, 1 / np.sqrt(d), (H, d)),
            "Wh": r.normal(0, 1 / np.sqrt(H), (H, H)), "b1": np.zeros(H),
            "U": r.normal(0, 1 / np.sqrt(H), (V, H)), "b2": np.zeros(V)}

def rnn_lm(p, X, Y, M, need_grad=True):
    B, T = X.shape; H = p["Wh"].shape[0]
    h = np.zeros((B, H)); hs, es, Ps = [h], [], []
    loss, ntok = 0.0, M.sum()
    for t in range(T):                                              # forward through time
        e = p["E"][X[:, t]]
        h = np.tanh(e @ p["We"].T + h @ p["Wh"].T + p["b1"])
        P = softmax(h @ p["U"].T + p["b2"])
        loss -= (np.log(P[np.arange(B), Y[:, t]] + 1e-12) * M[:, t]).sum()
        es.append(e); hs.append(h); Ps.append(P)
    loss /= ntok
    if not need_grad:
        return loss, Ps, hs
    g = {k: np.zeros_like(v) for k, v in p.items()}
    dh_next = np.zeros((B, H))                                      # gradient arriving from step t+1
    for t in reversed(range(T)):                                    # backward through time
        G = Ps[t].copy(); G[np.arange(B), Y[:, t]] -= 1; G *= M[:, t][:, None] / ntok
        g["U"] += G.T @ hs[t + 1]; g["b2"] += G.sum(0)
        dh = G @ p["U"] + dh_next                                    # from the output AND from the future
        dz = dh * (1 - hs[t + 1] ** 2)
        g["We"] += dz.T @ es[t]; g["Wh"] += dz.T @ hs[t]; g["b1"] += dz.sum(0)   # sum over time steps
        np.add.at(g["E"], X[:, t], dz @ p["We"])
        dh_next = dz @ p["Wh"]
    return loss, g

# Gradient check on a tiny model
p_chk = init_rnn(V, d=4, H=5, seed=1)
Xc, Yc, Mc = batchify(train_data[:3])
_, g_chk = rnn_lm(p_chk, Xc, Yc, Mc)
r = np.random.default_rng(2)
for k in p_chk:
    idx = (Xc[0, 2], 1) if k == "E" else tuple(r.integers(0, s) for s in p_chk[k].shape)
    old = p_chk[k][idx]
    p_chk[k][idx] = old + 1e-5; lp = rnn_lm(p_chk, Xc, Yc, Mc, False)[0]
    p_chk[k][idx] = old - 1e-5; lm_ = rnn_lm(p_chk, Xc, Yc, Mc, False)[0]
    p_chk[k][idx] = old
    print(f"dJ/d{k}{idx}: analytic {g_chk[k][idx]: .6e}   numerical {(lp - lm_) / 2e-5: .6e}")

**Verifying "the gradient w.r.t. a repeated weight is the sum of the gradients w.r.t. each time it appears."** Below, we unroll the RNN with a *separate copy* of $W_h$ at every timestep, compute the gradient with respect to each copy numerically, and check that they sum to the BPTT gradient.

In [ ]:
def rnn_loss_per_step_Wh(p, Wh_list, X, Y, M):
    B, T = X.shape; h = np.zeros((B, p["Wh"].shape[0])); loss = 0.0
    for t in range(T):
        h = np.tanh(p["E"][X[:, t]] @ p["We"].T + h @ Wh_list[t].T + p["b1"])
        P = softmax(h @ p["U"].T + p["b2"])
        loss -= (np.log(P[np.arange(B), Y[:, t]] + 1e-12) * M[:, t]).sum()
    return loss / M.sum()

T_ = Xc.shape[1]
i, j = 1, 3                                         # look at one entry of W_h
per_step = []
for step in range(T_):
    Ws = [p_chk["Wh"].copy() for _ in range(T_)]
    Ws[step][i, j] += 1e-5; lp = rnn_loss_per_step_Wh(p_chk, Ws, Xc, Yc, Mc)
    Ws[step][i, j] -= 2e-5; lm_ = rnn_loss_per_step_Wh(p_chk, Ws, Xc, Yc, Mc)
    per_step.append((lp - lm_) / 2e-5)
print("dJ/dW_h[1,3] at each timestep:", np.array(per_step).round(6))
print(f"sum over timesteps = {sum(per_step):.6e}   BPTT gradient = {g_chk['Wh'][i, j]:.6e}")

### 4.4 Training the RNN-LM on the agreement corpus
We use Adam and **gradient clipping** (§6.4 explains why clipping matters for RNNs). We track test perplexity and agreement accuracy after every epoch.

In [ ]:
def clip_grads(g, max_norm=5.0):
    norm = math.sqrt(sum((v ** 2).sum() for v in g.values()))
    if norm > max_norm:
        for k in g:
            g[k] *= max_norm / norm
    return norm

def rnn_agreement(p, sentences):
    X, Y, M = batchify(sentences)
    _, Ps, _ = rnn_lm(p, X, Y, M, False)
    acc = collections.defaultdict(list)
    for b, (s, sp, vp, pl, vpair) in enumerate(sentences):
        P = Ps[vp][b]                     # input position vp is word s[vp-1]; it predicts s[vp] = the verb
        acc[bucket(vp - sp)].append(P[w2i[vpair[pl]]] > P[w2i[vpair[1 - pl]]])
    return {k: round(float(np.mean(v)), 3) for k, v in sorted(acc.items())}

p_rnn = init_rnn(V)
opt = Adam(p_rnn)
r = np.random.default_rng(0)
history = []
t0 = time.time()
for epoch in range(12):
    order = r.permutation(len(train_data))
    for k in range(0, len(order), 64):
        X, Y, M = batchify([train_data[i] for i in order[k:k + 64]])
        _, g = rnn_lm(p_rnn, X, Y, M)
        clip_grads(g)
        opt.step(p_rnn, g)
    Xt, Yt, Mt = batchify(test_data)
    ppl = math.exp(rnn_lm(p_rnn, Xt, Yt, Mt, False)[0])
    agr = rnn_agreement(p_rnn, test_data)
    history.append((ppl, agr))
    print(f"epoch {epoch + 1:2d}: test perplexity {ppl:.3f}   agreement {agr}")
print(f"({time.time() - t0:.0f}s)")

In [ ]:
buckets = ["1", "2-6", "7-10", "11+"]
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for bk in buckets:
    axes[0].plot(range(1, len(history) + 1), [h[1][bk] for h in history], "o-", label=f"distance {bk}")
axes[0].axhline(0.5, color="grey", ls="--"); axes[0].set_xlabel("epoch"); axes[0].set_ylabel("agreement accuracy")
axes[0].set_title("RNN: agreement accuracy during training"); axes[0].legend()

models = {"bigram": agreement_ngram(ngrams[2], test_data), "trigram": agreement_ngram(ngrams[3], test_data),
          "window (k=3)": agreement_window(p_win, test_data), "RNN": history[-1][1]}
xs = np.arange(len(buckets))
for i_, (name, a) in enumerate(models.items()):
    axes[1].bar(xs + 0.2 * i_, [a[bk] for bk in buckets], width=0.2, label=name)
axes[1].axhline(0.5, color="grey", ls="--"); axes[1].set_xticks(xs + 0.3); axes[1].set_xticklabels(buckets)
axes[1].set_xlabel("subject-verb distance"); axes[1].set_title("Agreement accuracy by model"); axes[1].legend()
plt.tight_layout(); plt.show()

Two things to notice:
1. **The RNN solves long-distance agreement** that no fixed-context model can, because its hidden state can carry the subject's number for as long as needed.
2. **The learning curve has a plateau.** For several epochs, agreement at long distances sits at chance, then it jumps. Early in training, the error signal at the verb must travel back many steps to reach the subject, and it arrives weak and noisy (the **vanishing gradient** problem, §6). Short-distance examples are learned first, and they bootstrap a "number" feature that the network then learns to keep.

---
## 5. Generating with an RNN-LM, and evaluating LMs

### 5.1 Generating "roll-outs"
Just like an n-gram LM, you can use an RNN-LM to **generate text by repeated sampling**. The **sampled output becomes the next step's input**:

`<s>` → sample *my* → sample *favorite* → sample *season* → sample *is* → sample *spring* → sample `</s>`

The slides show fun examples: RNN-LMs trained on Obama speeches, Harry Potter, recipes, and paint color names (that last one is a **character-level** RNN-LM, predicting the next *character*).

🔎 **Decoding knobs you'll use with every LLM:**
* **Temperature** $\tau$: sample from $\text{softmax}(\text{logits}/\tau)$. $\tau<1$ sharpens (safer, more repetitive), $\tau>1$ flattens (more diverse, more errors), $\tau\to0$ = greedy argmax.
* **Top-$k$ sampling:** sample only from the $k$ most likely tokens.
* **Nucleus (top-$p$) sampling** (Holtzman et al. 2020): sample from the smallest set of tokens whose cumulative probability $\geq p$.

In [ ]:
def sample_rnn(p, temperature=1.0, top_k=None, max_len=30, rng=np.random.default_rng(0)):
    h = np.zeros(p["Wh"].shape[0]); w = w2i["<s>"]; out = []
    for _ in range(max_len):
        h = np.tanh(p["We"] @ p["E"][w] + p["Wh"] @ h + p["b1"])
        logits = (p["U"] @ h + p["b2"]) / temperature
        if top_k is not None:
            cutoff = np.sort(logits)[-top_k]
            logits = np.where(logits >= cutoff, logits, -np.inf)
        w = rng.choice(V, p=softmax(logits))
        if vocab[w] == "</s>":
            break
        out.append(vocab[w])
    return out

for tau in [0.5, 1.0, 1.5]:
    gen = [sample_rnn(p_rnn, tau, rng=np.random.default_rng(i)) for i in range(400)]
    viol = [v for v in map(agreement_violation, gen) if v is not None]
    print(f"temperature {tau}: e.g. '{' '.join(gen[0])}'")
    print(f"      agreement violations {np.mean(viol):.1%}, mean length {np.mean([len(g) for g in gen]):.1f}")
print("\n(the trigram model's samples violated agreement much more often; see Sec. 2.4)")

### 5.2 Evaluating language models: perplexity
The standard evaluation metric for language models is **perplexity**:
$$\text{perplexity} = \prod_{t=1}^{T}\left(\frac{1}{P_{\text{LM}}\big(x^{(t+1)}\mid x^{(t)},\dots,x^{(1)}\big)}\right)^{1/T}$$
the **inverse probability of the corpus** according to the LM, **normalized by the number of words**. This is equal to the **exponential of the cross-entropy loss** $J(\theta)$:
$$= \prod_{t=1}^{T}\left(\frac{1}{\hat y^{(t)}_{x_{t+1}}}\right)^{1/T} = \exp\left(\frac{1}{T}\sum_{t=1}^{T} -\log\hat y^{(t)}_{x_{t+1}}\right) = \exp(J(\theta))$$

**Lower perplexity is better!** Intuition: a perplexity of $k$ means the model is, on average, as uncertain as if it were choosing uniformly among $k$ words. (🔎 Equivalently, $\log_2(\text{perplexity})$ is the average number of **bits** per word the model needs to encode the text.) RNNs greatly improved perplexity over what came before; the slides' chart (Facebook's billion-word benchmark) shows perplexity dropping steadily from n-grams to increasingly complex RNNs.

⚠️ Perplexities are only comparable with the **same vocabulary and tokenization** on the same test set. (A character-level and a word-level model can't be compared directly.)

In [ ]:
Xt, Yt, Mt = batchify(test_data)
results = {f"{n}-gram (add-0.01)": perplexity_ngram(lm, test_data) for n, lm in ngrams.items()}
results["fixed window (k=3)"] = math.exp(window_lm_fb(p_win, Xw_te, Yw_te, False)[0])
results["RNN"] = math.exp(rnn_lm(p_rnn, Xt, Yt, Mt, False)[0])
results["uniform guessing"] = V
for name, ppl in results.items():
    print(f"{name:22s} perplexity {ppl:7.3f}   ({math.log2(ppl):.2f} bits/word)")

The perplexity differences look small because most words in our toy language are easy to predict (*the* after *that*, etc.). The verb-agreement decisions are a handful of tokens per sentence. **A lesson about evaluation:** an aggregate metric like perplexity can hide exactly the capability you care about. Targeted evaluations (like our agreement test, or the BLiMP benchmark for real LMs) complement it.

---
## 6. Problems with RNNs: vanishing and exploding gradients

### 6.1 Vanishing gradient intuition
Consider the gradient of the loss at step 4 with respect to the hidden state at step 1. By the chain rule:
$$\frac{\partial J^{(4)}}{\partial h^{(1)}} = \frac{\partial h^{(2)}}{\partial h^{(1)}}\times\frac{\partial h^{(3)}}{\partial h^{(2)}}\times\frac{\partial h^{(4)}}{\partial h^{(3)}}\times\frac{\partial J^{(4)}}{\partial h^{(4)}}$$
**What happens if these factors are small?** **Vanishing gradient problem:** when they are small, the gradient signal gets smaller and smaller as it backpropagates further.

### 6.2 Proof sketch (linear case) (Pascanu, Mikolov & Bengio 2013)
Recall $h^{(t)} = \sigma(W_h h^{(t-1)} + W_x x^{(t)} + b_1)$. What if $\sigma$ were the identity function, $\sigma(x) = x$? Then
$$\frac{\partial h^{(t)}}{\partial h^{(t-1)}} = \text{diag}\big(\sigma'(\cdot)\big)W_h = I\,W_h = W_h$$
Consider the gradient of the loss $J^{(i)}$ on step $i$ with respect to the hidden state $h^{(j)}$ on some previous step $j$. Let $\ell = i - j$:
$$\frac{\partial J^{(i)}}{\partial h^{(j)}} = \frac{\partial J^{(i)}}{\partial h^{(i)}}\prod_{j<t\leq i}\frac{\partial h^{(t)}}{\partial h^{(t-1)}} = \frac{\partial J^{(i)}}{\partial h^{(i)}}\,W_h^{\ell}$$
If $W_h$ is "small", this term gets **exponentially problematic** as $\ell$ becomes large.

**What's wrong with $W_h^\ell$?** Consider if the eigenvalues of $W_h$ are all less than 1 in magnitude: $\lambda_1, \dots, \lambda_n$ with $|\lambda_i| < 1$, and eigenvectors $q_1, \dots, q_n$. We can write the gradient using the eigenvectors as a basis:
$$\frac{\partial J^{(i)}}{\partial h^{(i)}}W_h^\ell = \sum_{k=1}^{n}c_k\,\lambda_k^{\ell}\,q_k \approx 0 \quad\text{for large }\ell$$
It **approaches 0 as $\ell$ grows**, so the gradient vanishes. (If some $|\lambda_k| > 1$, it explodes instead.)

**What about nonlinear activations?** Pretty much the same thing, except the proof requires $\lambda_i < \gamma$ for some $\gamma$ dependent on the dimensionality and on $\sigma$ (e.g., $|\tanh'|\leq1$, $|\sigma'|\leq 1/4$). It's a **sufficient but not necessary** condition.

Let's measure it. Below, a random $W_h$ is rescaled to spectral radius $\rho$ (largest $|\lambda|$), and we compute $\|\partial h^{(t)}/\partial h^{(0)}\|$ (through the Jacobian product) for a linear RNN and a tanh RNN:

In [ ]:
def jacobian_norms(rho, activation, T=60, H=64, seed=0):
    r = np.random.default_rng(seed)
    W = r.normal(size=(H, H)); W *= rho / np.max(np.abs(np.linalg.eigvals(W)))
    Wx = r.normal(0, 1 / np.sqrt(H), (H, H))
    h = np.zeros(H); J = np.eye(H); norms = []
    for t in range(T):
        z = W @ h + Wx @ r.normal(size=H)
        if activation == "linear":
            h, D = z, np.ones(H)
        else:
            h, D = np.tanh(z), 1 - np.tanh(z) ** 2
        J = (D[:, None] * W) @ J                      # dh_t/dh_0 = diag(f'(z_t)) W ... diag(f'(z_1)) W
        norms.append(np.linalg.norm(J, 2))
    return norms

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, act in zip(axes, ["linear", "tanh"]):
    for rho in [0.8, 1.0, 1.2, 2.0]:
        ax.semilogy(range(1, 61), jacobian_norms(rho, act), label=f"spectral radius {rho}")
    ax.set_title(f"{act} RNN: ||d h(t) / d h(0)||"); ax.set_xlabel("t (steps back)"); ax.legend()
plt.tight_layout(); plt.show()

* **Linear RNN:** exactly the proof: $\rho<1$ vanishes exponentially, $\rho>1$ explodes exponentially, $\rho=1$ is the knife edge.
* **tanh RNN:** large $\rho$ no longer explodes forever, because the tanh **saturates** ($\tanh'\to0$ when units are pushed to ±1), which *also* kills the gradient. So for tanh RNNs vanishing is the default failure mode, from both sides. (At $\rho\approx1$ with small activations, gradients survive much longer. This is why our RNN, initialized with $W_h\sim N(0, 1/H)$, i.e. $\rho\approx1$, could learn agreement at all.)

### 6.3 Why is vanishing gradient a problem?
The gradient signal from far away is lost because it's much smaller than the gradient signal from close by. So model weights are updated only with respect to **near effects**, not **long-term effects**.

**Effect on an RNN-LM.** *"When she tried to print her **tickets**, she found that the printer was out of toner. She went to the stationery store to buy more toner. It was very overpriced. After installing the toner into the printer, she finally printed her \_\_\_\_\_\_\_\_."* To learn from this example, the RNN-LM needs to model the dependency between *tickets* on the 7th step and the target word *tickets* at the end. If the gradient is small, the model can't learn this dependency, and so it can't predict similar long-distance dependencies at test time.

### 6.4 Why is exploding gradient a problem?
If the gradient becomes too big, the SGD update step becomes too big: $\theta^{\text{new}} = \theta^{\text{old}} - \alpha\nabla_\theta J(\theta)$. This can cause **bad updates**: we take too large a step and reach a weird and bad parameter configuration (with large loss). *"You think you've found a hill to climb, but suddenly you're in Iowa."* In the worst case, this will result in **Inf or NaN** in your network, and you have to restart training from an earlier checkpoint.

### 6.5 Gradient clipping: the solution for exploding gradients
**If the norm of the gradient is greater than some threshold, scale it down before applying the SGD update:**
$$\hat g \leftarrow \nabla_\theta J;\qquad \text{if } \|\hat g\| \geq \text{threshold}: \quad \hat g \leftarrow \frac{\text{threshold}}{\|\hat g\|}\hat g$$
**Intuition:** take a step in the **same direction**, but a smaller step. In practice, remembering to clip gradients is important, but exploding gradients are an *easy* problem to solve. (Our `clip_grads` above does exactly this, across *all* parameters jointly.)

**The "cliff" (Pascanu et al. 2013).** RNN loss surfaces can contain nearly vertical walls where the dynamics change qualitatively (a *bifurcation*). Here's a one-unit RNN $h_t = w\tanh(h_{t-1}) + b$ run for 30 steps, with loss $(h_{30} - 1.5)^2$, plotted as a function of the recurrent weight $w$:

In [ ]:
T_C, B_C, H0_C, TARGET_C = 30, -0.2, 0.5, 1.5

def cliff_loss_grad(w):
    h, hs = H0_C, [H0_C]
    for _ in range(T_C):
        h = w * np.tanh(h) + B_C; hs.append(h)
    L = (h - TARGET_C) ** 2
    dh, dw = 2 * (h - TARGET_C), 0.0
    for t in reversed(range(T_C)):                     # BPTT for one scalar unit
        dw += dh * np.tanh(hs[t]); dh = dh * w * (1 - np.tanh(hs[t]) ** 2)
    return L, dw

ws = np.linspace(0.8, 2.6, 4000)
Ls = [cliff_loss_grad(w)[0] for w in ws]
w_cliff = ws[np.argmax(np.abs(np.diff(Ls)))]
print(f"cliff near w = {w_cliff:.4f}; gradient there = {cliff_loss_grad(1.5140)[1]:.0f}")

def descend(w, clip, steps=20, lr=0.05):
    traj = [w]
    for _ in range(steps):
        _, g = cliff_loss_grad(w)
        if clip is not None and abs(g) > clip:
            g = clip * np.sign(g)
        w = w - lr * g; traj.append(w)
    return traj

for clip in [None, 1.0]:
    tr = descend(1.5140, clip)
    print(f"clip={clip}: w after 1 step = {tr[1]:.3f}; after 20 steps w = {tr[-1]:.3f}, loss = {cliff_loss_grad(tr[-1])[0]:.4f}")

plt.figure(figsize=(8, 4))
plt.plot(ws, Ls, color="black")
tr = descend(1.5140, 1.0)
plt.plot(tr, [cliff_loss_grad(w)[0] for w in tr], "o-", color="green", ms=3, label="with clipping")
plt.axvline(w_cliff, color="red", ls=":", label="cliff")
plt.xlabel("recurrent weight w"); plt.ylabel("loss"); plt.legend(); plt.title("An RNN loss 'cliff' (exploding gradient)")
plt.show()

Without clipping, a single step from the cliff wall throws $w$ hundreds of units away: the gradient there is in the thousands. (In this 1-D toy it can eventually walk back; in a network with millions of parameters, a step like that wrecks everything learned so far.) With clipping, the step keeps the same *direction* and the optimizer descends calmly to the minimum.

### 6.6 How to fix the vanishing gradient problem?
The main problem is that it's **too difficult for the RNN to learn to preserve information over many timesteps**: in a vanilla RNN, the hidden state is constantly being **rewritten**, $h^{(t)} = \sigma(W_h h^{(t-1)} + \dots)$.
* First off, next time: how about an RNN with **separate memory which is added to**? → **LSTMs** (L6)
* And then: creating more **direct and linear pass-through connections** in the model → **attention** (L7), **residual connections** (L6, L8), etc.

---
## 7. Recap, and other uses of RNNs

* **Language model:** a system that predicts the next word.
* **Recurrent neural network:** a family of neural networks that take sequential input of any length, apply the same weights on each step, and can optionally produce output on each step.
* **Recurrent neural network ≠ language model.** RNNs are a great way to build an LM (despite some problems), but RNNs are useful for much more, and there are other models for building LMs (esp. Transformers!).

**Why should we care about language modeling?**
* *Old answer:* it's a benchmark task that measures our progress on predicting language use, and a subcomponent of many NLP tasks, especially those generating text or estimating its probability: predictive typing, speech recognition, handwriting recognition, spelling/grammar correction, authorship identification, machine translation, summarization, dialogue.
* *New answer:* **everything in NLP has now been rebuilt upon language modeling!** GPT-3, GPT-4, Claude, Gemini: all LMs. We can now instruct LMs to do language understanding and reasoning tasks for us.

**Other RNN uses:**
* **Sequence tagging** (POS tagging, NER): an output at every step. *the/DT startled/JJ cat/NN knocked/VBN over/IN the/DT vase/NN*
* **Sentence classification** (e.g., sentiment): compute a **sentence encoding**, then classify. Basic way: use the **final hidden state**. Usually better: take the **element-wise max or mean of all hidden states**.
* **Encoder module** (QA, MT, many others): the hidden states represent the question/source and feed a larger neural system.
* **Conditional language models** (speech recognition, MT, summarization): an RNN-LM that generates text *conditioned on* something else (audio, a source sentence). → L6

### 7.1 🔎 Looking inside: a diagnostic probe
Is the subject's number really stored in the hidden state? A standard interpretability technique (Hupkes et al. 2018; "probing") is to train a simple **linear classifier** to predict a property from the representations. We read the RNN's hidden state at the word just before the verb and try to predict whether the subject was plural. Compare: the same probe on a "bag of the previous 3 words" representation.

In [ ]:
def hidden_before_verb(p, sentences):
    X, Y, M = batchify(sentences)
    _, _, hs = rnn_lm(p, X, Y, M, False)
    # hs[0] is the initial state, so hs[t+1] is the state after input position t.
    # Input position vp holds word s[vp-1] (inputs are shifted by <s>), so the state that predicts the verb is hs[vp+1].
    return np.array([hs[vp + 1][b] for b, (_, _, vp, _, _) in enumerate(sentences)])

def train_logreg(F, y, steps=500, lr=0.5):
    w, b = np.zeros(F.shape[1]), 0.0
    for _ in range(steps):
        pr = 1 / (1 + np.exp(-(F @ w + b)))
        w -= lr * F.T @ (pr - y) / len(y); b -= lr * np.mean(pr - y)
    return w, b

y_tr = np.array([pl for *_, pl, _ in train_data[:3000]], float)
y_te = np.array([pl for *_, pl, _ in test_data], float)
F_tr, F_te = hidden_before_verb(p_rnn, train_data[:3000]), hidden_before_verb(p_rnn, test_data)
mu, sd = F_tr.mean(0), F_tr.std(0) + 1e-8                 # standardize features: probes train much better
w_, b_ = train_logreg((F_tr - mu) / sd, y_tr, steps=3000)
acc_rnn = ((((F_te - mu) / sd @ w_ + b_) > 0) == y_te).mean()

def last3_bag(sentences):
    F = np.zeros((len(sentences), V))
    for i, (s, _, vp, _, _) in enumerate(sentences):
        for w in s[max(0, vp - 3):vp]:
            F[i, w2i[w]] = 1
    return F
w3, b3 = train_logreg(last3_bag(train_data[:3000]), y_tr, steps=3000)
acc_bag = (((last3_bag(test_data) @ w3 + b3) > 0) == y_te).mean()
print(f"linear probe for 'subject is plural' from the RNN hidden state:   {acc_rnn:.1%}")
print(f"linear probe from the last 3 words only (what a 4-gram can see): {acc_bag:.1%}")

The subject's number is **linearly decodable** from the RNN's hidden state at the verb, even many words later. A memory of the last few words can't do this.

### 7.2 Terminology, and a look forward
By the end of the course you'll understand phrases like *"stacked bidirectional LSTMs with residual connections and self-attention."* The RNN in this lecture is the **simple / vanilla / Elman RNN**. Next lecture: other RNN flavors like the **LSTM** and **GRU**, and **multi-layer** RNNs.

---
## 8. The same in PyTorch

`nn.RNN` implements exactly our recurrence ($h_t = \tanh(W_{ih}x_t + b_{ih} + W_{hh}h_{t-1} + b_{hh})$), batched and fast. `ignore_index` in the loss plays the role of our mask. Note `clip_grad_norm_`.

In [ ]:
# [PyTorch] RNN language model on the agreement corpus.
import torch
import torch.nn as nn
import torch.nn.functional as F

class RNNLM(nn.Module):
    def __init__(self, vocab_size, d=32, hidden=64, cell="rnn"):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, d)
        rnn_cls = {"rnn": nn.RNN, "lstm": nn.LSTM, "gru": nn.GRU}[cell]
        self.rnn = rnn_cls(d, hidden, batch_first=True)
        self.out = nn.Linear(hidden, vocab_size)
    def forward(self, x, state=None):
        h, state = self.rnn(self.emb(x), state)   # h: (B, T, hidden)
        return self.out(h), state

PAD = -100                                         # cross_entropy ignores this target value
def to_torch(batch):
    X, Y, M = batchify(batch)
    Y = np.where(M > 0, Y, PAD)
    return torch.tensor(X), torch.tensor(Y)

torch.manual_seed(0)
model = RNNLM(V)
opt_t = torch.optim.Adam(model.parameters(), lr=2e-3)
for epoch in range(12):
    model.train()
    perm = np.random.default_rng(epoch).permutation(len(train_data))
    for k in range(0, len(perm), 64):
        X, Y = to_torch([train_data[i] for i in perm[k:k + 64]])
        logits, _ = model(X)
        loss = F.cross_entropy(logits.reshape(-1, V), Y.reshape(-1), ignore_index=PAD)
        opt_t.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        opt_t.step()
    model.eval()
    with torch.no_grad():
        X, Y = to_torch(test_data)
        logits, _ = model(X)
        ppl = math.exp(F.cross_entropy(logits.reshape(-1, V), Y.reshape(-1), ignore_index=PAD).item())
        P = logits.softmax(-1).numpy()
    acc = collections.defaultdict(list)
    for b, (s, sp, vp, pl, vpair) in enumerate(test_data):
        acc[bucket(vp - sp)].append(P[b, vp, w2i[vpair[pl]]] > P[b, vp, w2i[vpair[1 - pl]]])
    print(f"epoch {epoch + 1}: perplexity {ppl:.3f}  agreement {({k: round(float(np.mean(v)), 3) for k, v in sorted(acc.items())})}")

### 8.1 Optional: a character-level RNN-LM on real text (Tiny Shakespeare)
To see the "fun" side of the lecture, train a character-level LSTM on Shakespeare (~1MB; a GPU makes this quick, a CPU takes a few minutes for a recognizable result). Watch the samples go from noise, to word-like strings, to plausible-looking verse.

In [ ]:
# [Optional] needs PyTorch + internet. Character-level language model on Tiny Shakespeare.
try:
    import urllib.request, torch, torch.nn as nn, torch.nn.functional as F
    url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
    text = urllib.request.urlopen(url, timeout=20).read().decode("utf-8")
    chars = sorted(set(text)); c2i = {c: i for i, c in enumerate(chars)}
    data_t = torch.tensor([c2i[c] for c in text])
    device = "cuda" if torch.cuda.is_available() else "cpu"

    class CharLM(nn.Module):
        def __init__(self, n, d=64, h=256):
            super().__init__()
            self.emb = nn.Embedding(n, d); self.rnn = nn.LSTM(d, h, num_layers=2, batch_first=True, dropout=0.2)
            self.out = nn.Linear(h, n)
        def forward(self, x, s=None):
            y, s = self.rnn(self.emb(x), s); return self.out(y), s

    m = CharLM(len(chars)).to(device); opt_c = torch.optim.Adam(m.parameters(), lr=3e-3)

    def sample_chars(n=300, temp=0.8, prompt="ROMEO:"):
        m.eval(); x = torch.tensor([[c2i[c] for c in prompt]], device=device); s = None; out = prompt
        with torch.no_grad():
            logits, s = m(x, s)
            for _ in range(n):
                c = torch.multinomial((logits[0, -1] / temp).softmax(-1), 1)
                out += chars[c.item()]; logits, s = m(c.view(1, 1), s)
        m.train(); return out

    seq, bs = 128, 64
    for step in range(1, 2001):
        ix = torch.randint(0, len(data_t) - seq - 1, (bs,))
        x = torch.stack([data_t[i:i + seq] for i in ix]).to(device)
        y = torch.stack([data_t[i + 1:i + seq + 1] for i in ix]).to(device)
        logits, _ = m(x)
        loss = F.cross_entropy(logits.reshape(-1, len(chars)), y.reshape(-1))
        opt_c.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(m.parameters(), 1.0); opt_c.step()
        if step % 500 == 0:
            print(f"--- step {step}, loss {loss.item():.3f}, perplexity per char {math.exp(loss.item()):.2f}\n{sample_chars()}\n")
except Exception as e:
    print("Skipping char-RNN demo:", e)

---
## 9. Summary

| Concept | Key point |
|---|---|
| Dropout | Zero inputs with prob. $p$ in training; scale to preserve expectations; prevents co-adaptation |
| Initialization | Small random (break symmetry); Xavier $\text{Var} = 2/(n_{in}+n_{out})$ |
| Optimizers | SGD + LR schedule, or Adam/AdamW at ~1e-3 |
| Language model | $P(x^{(t+1)}\mid x^{(\le t)})$; chain rule gives $P(\text{text})$ |
| n-gram LM | Markov assumption + counts; sparsity (smoothing, backoff), storage; can't see beyond $n-1$ words |
| Fixed-window neural LM | No sparsity, but window too small and no weight sharing across positions |
| RNN | $h^{(t)} = \sigma(W_h h^{(t-1)} + W_e e^{(t)} + b)$, same weights every step, any length |
| Training | Teacher forcing; average per-step cross-entropy; batches of sentences |
| BPTT | Gradient for a repeated weight = sum over the times it's used; truncate in practice |
| Perplexity | $\exp(\text{cross-entropy})$; lower is better; only comparable with the same tokenization |
| Vanishing / exploding | Products of Jacobians $\approx W_h^\ell$; eigenvalues < 1 vanish, > 1 explode |
| Gradient clipping | Rescale the gradient if its norm exceeds a threshold: same direction, smaller step |

### Exercises
1. Implement **linear interpolation** smoothing for the trigram model ($\lambda_3P_3+\lambda_2P_2+\lambda_1P_1$), tune the $\lambda$s on held-out data, and compare perplexity.
2. Increase the window of the fixed-window LM to $k=10$. Does agreement improve at distances ≤ 9? At 11+?
3. Implement **truncated BPTT**: stop `dh_next` after 5 steps. What happens to agreement accuracy at long distances, and why?
4. Train the RNN with `max_mods=6` (longer distances) and compare the plateau length to `max_mods=3`.
5. Remove gradient clipping and train with a 10× higher learning rate. Do you see loss spikes?
6. Implement **nucleus (top-$p$) sampling** in `sample_rnn`.
7. Probe the hidden state at *every* position between the subject and the verb. Does the probe accuracy dip inside relative clauses?

### References
* Bengio, Ducharme, Vincent & Jauvin 2003, *A Neural Probabilistic Language Model*.
* Mikolov et al. 2010, *Recurrent Neural Network Based Language Model*.
* Pascanu, Mikolov & Bengio 2013, *On the Difficulty of Training Recurrent Neural Networks*.
* Linzen, Dupoux & Goldberg 2016, *Assessing the Ability of LSTMs to Learn Syntax-Sensitive Dependencies*.
* Srivastava et al. 2014, *Dropout: A Simple Way to Prevent Neural Networks from Overfitting*.
* Karpathy 2015, *The Unreasonable Effectiveness of Recurrent Neural Networks* (blog).
* Jurafsky & Martin, *Speech and Language Processing* (3rd ed.), Ch. 3 (n-gram LMs) and Ch. 8 (RNNs).

**Next:** Lecture 6, LSTMs, bidirectional and multi-layer RNNs, and neural machine translation.